# Downloading Youtube videos

In [1]:
from yt_dlp import YoutubeDL
import os

In [2]:
url = "https://www.youtube.com/watch?v=z3U0udLH974"
output_folder = "downloads"

# Step 1: Download video only (H.264 MP4)
video_opts = {
    'format': 'bestvideo[ext=mp4]',
    'outtmpl': os.path.join(output_folder, '%(title)s.video.mp4')
}

with YoutubeDL(video_opts) as ydl:
    ydl.download([url])

[youtube] Extracting URL: https://www.youtube.com/watch?v=z3U0udLH974
[youtube] z3U0udLH974: Downloading webpage


[youtube] z3U0udLH974: Downloading android sdkless player API JSON
[youtube] z3U0udLH974: Downloading web safari player API JSON


[youtube] z3U0udLH974: Downloading m3u8 information


[info] z3U0udLH974: Downloading 1 format(s): 397
[download] downloads/The two talking cats.video.mp4 has already been downloaded
[download] 100% of    1.06MiB


In [3]:
import yt_dlp
import os

url = "https://www.youtube.com/watch?v=z3U0udLH974"
output_folder = "downloads"
os.makedirs(output_folder, exist_ok=True)

ydl_opts = {
    'format': 'bestaudio/best',  # audio only
    'outtmpl': os.path.join(output_folder, '%(title)s.%(ext)s'),
    'postprocessors': [
        {
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',   # change to 'wav' for WAV
            'preferredquality': '192', # kbps for mp3
        }
    ],
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

print(f"✅ Audio saved in {output_folder} as MP3")

[youtube] Extracting URL: https://www.youtube.com/watch?v=z3U0udLH974
[youtube] z3U0udLH974: Downloading webpage


[youtube] z3U0udLH974: Downloading android sdkless player API JSON
[youtube] z3U0udLH974: Downloading web safari player API JSON


[youtube] z3U0udLH974: Downloading m3u8 information


[info] z3U0udLH974: Downloading 1 format(s): 251
[download] Destination: downloads/The two talking cats.webm
[download] 100% of  633.36KiB in 00:00:00 at 3.21MiB/s   
[ExtractAudio] Destination: downloads/The two talking cats.mp3
Deleting original file downloads/The two talking cats.webm (pass -k to keep)
✅ Audio saved in downloads as MP3


# Stealing YouTube videos💃

In [27]:
# First, code for downloading video

In [3]:
def download_video(video_idx: str, output_folder: str = "downloads") -> dict:
    """
    Download a YouTube video and its audio (MP3) safely with progress bar.
    Skips the video if any step fails.
    """
    os.makedirs(output_folder, exist_ok=True)
    url = f"https://www.youtube.com/watch?v={video_idx}"

    # Options for yt-dlp: download best video and audio separately
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio/best',  # separate video & audio
        'outtmpl': os.path.join(output_folder, '%(title)s.%(ext)s'),
        'merge_output_format': 'mp4',
        'progress_hooks': [],  # we will use tqdm manually
        'quiet': True,
        'no_warnings': True,  # <-- suppress warnings
    }

    try:
        with YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            title = info.get('title', 'unknown')
            video_id = info.get('id', video_idx)
        
        # Convert audio to MP3
        audio_opts = {
            'format': 'bestaudio/best',
            'outtmpl': os.path.join(output_folder, f'{title}.mp3'),
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
            'quiet': True,
            'no_warnings': True,  # <-- suppress warnings
        }

        with YoutubeDL(audio_opts) as ydl:
            ydl.download([url])
        
        return {
            "video_id": video_id,
            "video_title": title
        }

    except Exception as e:
        return {}

In [6]:
download_video("sZkB11pO9R8", output_folder="downloads/VideoData")

✅ Downloaded 'How a cat works' (sZkB11pO9R8) to downloads/VideoData


In [7]:
# now getting all videos to download

In [18]:
import pandas as pd
import ast

In [15]:
import pandas as pd

balanced_videos_path = "data/video-data/balanced_train_segments.csv"

# Read CSV with Python engine and comma as delimiter
df = pd.read_csv(
    balanced_videos_path, 
    engine='python',   # better for messy CSVs
    skipinitialspace=True,
    quotechar='"'      # respect quoted strings
)

# Remove leading '--' in the YTID column
df['YTID'] = df['YTID'].str.lstrip('-')

# Convert last column (positive_labels) from string to list
df['positive_labels'] = df['positive_labels'].apply(
    lambda s: [x.strip() for x in s.strip('"').split(',')]
)

print(df.head())

        YTID  start_seconds  end_seconds                 positive_labels
0  PJHxphWEs           30.0         40.0          [/m/09x0r, /t/dd00088]
1  ZhevVpy1s           50.0         60.0                     [/m/012xff]
2  aE2O5G5WE            0.0         10.0  [/m/03fwl, /m/04rlf, /m/09x0r]
3  aO5cdqSAg           30.0         40.0        [/t/dd00003, /t/dd00005]
4  aaILOrkII          200.0        210.0          [/m/032s66, /m/073cg4]


In [16]:
# 2️⃣ Load the mapping CSV
label_map_path = "data/video-data/class_labels_indices.csv"
label_df = pd.read_csv(label_map_path)

# Build a dictionary: mid -> display_name
label_dict = dict(zip(label_df['mid'], label_df['display_name']))

# 3️⃣ Map each list of positive_labels to their display names
df['positive_labels_names'] = df['positive_labels'].apply(
    lambda lst: [label_dict.get(mid, mid) for mid in lst]
)

In [17]:
df.sample(5)

,YTID,start_seconds,end_seconds,positive_labels,positive_labels_names
17768,lpHa-S_x1LA,350.0,360.0,"[/m/01p970, /m/026t6, /m/0l14md]","[Tabla, Drum, Percussion]"
7909,J3W0GUqDDag,100.0,110.0,"[/m/025_jnm, /m/09x0r]","[Finger snapping, Speech]"
8343,KL0TWPmezrY,10.0,20.0,"[/m/04rlf, /m/07p6fty, /m/09x0r]","[Music, Shout, Speech]"
18340,nbb8vmfjHJs,220.0,230.0,"[/m/03lty, /m/04rlf, /m/06by7, /t/dd00036]","[Heavy metal, Music, Rock music, Angry music]"
4838,AtdCLd4StZk,130.0,140.0,"[/m/016cjb, /m/04rlf]","[Gospel music, Music]"


In [18]:
df.shape

(22160, 5)

In [44]:
# Filter rows where 'positive_labels_names' contains 'Car' (case-insensitive)
cat_rows = df[df['positive_labels_names'].apply(
    lambda labels: any(label.lower() == "cat" for label in labels)
)]

cat_rows = cat_rows[cat_rows['YTID'].str.len() == 11]

cat_rows.shape

(112, 5)

In [45]:
cat_rows.sample(5)

,YTID,start_seconds,end_seconds,positive_labels,positive_labels_names
1386,29x5I94EBMQ,0.0,9.0,"[/m/01yrx, /m/04rlf, /m/068hy, /m/07r81j2, /m/...","[Cat, Music, Domestic animals, pets, Caterwaul..."
21048,wEmEkLQDl0U,0.0,9.0,"[/m/01yrx, /m/07qrkrw, /m/0jbk]","[Cat, Meow, Animal]"
2643,58yb7qX_pv8,30.0,40.0,"[/m/01yrx, /m/04rlf, /m/068hy, /m/07qrkrw, /m/...","[Cat, Music, Domestic animals, pets, Meow, Cat..."
3252,6c-5b9sj35o,30.0,40.0,"[/m/01yrx, /m/068hy, /m/07qrkrw, /m/07r81j2, /...","[Cat, Domestic animals, pets, Meow, Caterwaul,..."
2909,5lNTPOv9fSE,510.0,520.0,"[/m/01yrx, /m/068hy, /m/07pdhp0, /m/0jbk, /t/d...","[Cat, Domestic animals, pets, Biting, Animal, ..."


In [34]:
from tqdm import tqdm
import json
import os

In [48]:
log_file = "downloading_logs.jsonl"

In [52]:
with open(log_file, "a") as f:
    for idx, row in tqdm(cat_rows.iterrows(), total=len(cat_rows)):
        video_id = str(row["YTID"]).strip("-").strip()  # remove leading '-' or spaces

        if not video_id:  # skip empty IDs
            print(f"⚠️ Skipping empty/truncated ID at row {idx}")
            continue

        try:
            video_info = download_video(
                video_idx=video_id,
                output_folder="downloads/VideoData"
            )

            if video_info is None:
                metadata = {
                    "video_id": row["YTID"],
                    "error" : f"⚠️ No info returned for {video_id}, skipping"
                }
                f.write(json.dumps(metadata, ensure_ascii=False) + "\n")
                continue

            metadata = {
                **video_info,
                "start_seconds": row["start_seconds"],
                "end_seconds": row["end_seconds"],
                "positive_labels": row["positive_labels_names"]
            }

            f.write(json.dumps(metadata, ensure_ascii=False) + "\n")

        except Exception as e:
            metadata = {
                "video_id": row["YTID"],
                "error" : e
            }
            f.write(json.dumps(metadata, ensure_ascii=False) + "\n")
            continue

  0%|          | 0/112 [00:00<?, ?it/s]

  2%|▏         | 2/112 [00:05<04:10,  2.28s/it]

  3%|▎         | 3/112 [00:07<04:43,  2.60s/it]

  4%|▎         | 4/112 [00:13<06:59,  3.88s/it]

  4%|▍         | 5/112 [00:18<07:26,  4.17s/it]

...

 96%|█████████▌| 107/112 [10:36<00:37,  7.45s/it]

 97%|█████████▋| 109/112 [10:40<00:13,  4.62s/it]

 98%|█████████▊| 110/112 [10:44<00:08,  4.27s/it]

 99%|█████████▉| 111/112 [10:49<00:04,  4.43s/it]

100%|██████████| 112/112 [10:56<00:00,  5.86s/it]


# Stealing YouTube videos but moooore🕺

In [2]:
import pandas as pd

balanced_videos_path = "data/video-data/unbalanced_train_segments.csv"

# Read CSV with Python engine and comma as delimiter
df = pd.read_csv(
    balanced_videos_path, 
    engine='python',   # better for messy CSVs
    skipinitialspace=True,
    quotechar='"'      # respect quoted strings
)

# Remove leading '--' in the YTID column
df['YTID'] = df['YTID'].str.lstrip('-')

# Convert last column (positive_labels) from string to list
df['positive_labels'] = df['positive_labels'].apply(
    lambda s: [x.strip() for x in s.strip('"').split(',')]
)

df.head()

,YTID,start_seconds,end_seconds,positive_labels
0,1_cCGK4M,0.0,10.0,"[/m/01g50p, /m/0284vy3, /m/06d_3, /m/07jdr, /m..."
1,2_BBVHAA,30.0,40.0,[/m/09x0r]
2,B_v8ZoBY,30.0,40.0,[/m/04rlf]
3,EDNidJUA,30.0,40.0,"[/m/02qldy, /m/02zsn, /m/05zppz, /m/09x0r]"
4,N4cFAE1A,21.0,31.0,"[/m/04rlf, /m/09x0r]"


In [3]:
# 2️⃣ Load the mapping CSV
label_map_path = "data/video-data/class_labels_indices.csv"
label_df = pd.read_csv(label_map_path)

# Build a dictionary: mid -> display_name
label_dict = dict(zip(label_df['mid'], label_df['display_name']))

# 3️⃣ Map each list of positive_labels to their display names
df['positive_labels_names'] = df['positive_labels'].apply(
    lambda lst: [label_dict.get(mid, mid) for mid in lst]
)

In [4]:
df.shape

(2041789, 5)

In [5]:
df.sample(3)

,YTID,start_seconds,end_seconds,positive_labels,positive_labels_names
361279,5Y_WHK8l86Q,30.0,40.0,"[/m/09x0r, /t/dd00125]","[Speech, Inside, small room]"
420268,6uDsMOHpCK8,30.0,40.0,"[/m/04rlf, /m/09x0r]","[Music, Speech]"
951239,M1IC55KxO3I,30.0,40.0,[/m/04rlf],[Music]


In [6]:
# Filter rows where 'positive_labels_names' contains 'Car' (case-insensitive)
cat_rows = df[df['positive_labels_names'].apply(
    lambda labels: any(label.lower() == "cat" for label in labels)
)]

cat_rows = cat_rows[cat_rows['YTID'].str.len() == 11]

cat_rows.shape

(3599, 5)

In [10]:
cat_rows.sample(3)

,YTID,start_seconds,end_seconds,positive_labels,positive_labels_names
1002773,Nh-0sVtY4p8,18.0,28.0,"[/m/01yrx, /m/068hy, /m/07qrkrw, /m/09x0r, /m/...","[Cat, Domestic animals, pets, Meow, Speech, An..."
1990816,xs5wYGHyqhE,60.0,70.0,"[/m/01yrx, /m/02yds9, /m/068hy, /m/0jbk]","[Cat, Purr, Domestic animals, pets, Animal]"
337634,52Pr5Bm3RUQ,6.0,16.0,"[/m/01yrx, /m/068hy, /m/07qrkrw, /m/0jbk]","[Cat, Domestic animals, pets, Meow, Animal]"


In [13]:
from tqdm import tqdm
import json
import os

In [11]:
log_file = "downloading_unbalanced_logs.jsonl"

In [19]:
with open(log_file, "a") as f:
    for idx, row in tqdm(cat_rows.iterrows(), total=len(cat_rows)):
        video_id = str(row["YTID"]).strip("-").strip()  # remove leading '-' or spaces

        if not video_id:  # skip empty IDs
            print(f"⚠️ Skipping empty/truncated ID at row {idx}")
            continue

        try:
            video_info = download_video(
                video_idx=video_id,
                output_folder="downloads/VideoDataUnbalanced"
            )

            if video_info is None:
                metadata = {
                    "video_id": row["YTID"],
                    "error" : f"⚠️ No info returned for {video_id}, skipping"
                }
                f.write(json.dumps(metadata, ensure_ascii=False) + "\n")
                continue

            metadata = {
                **video_info,
                "start_seconds": row["start_seconds"],
                "end_seconds": row["end_seconds"],
                "positive_labels": row["positive_labels_names"]
            }

            f.write(json.dumps(metadata, ensure_ascii=False) + "\n")

        except Exception as e:
            metadata = {
                "video_id": row["YTID"],
                "error" : e
            }
            f.write(json.dumps(metadata, ensure_ascii=False) + "\n")
            continue

  0%|          | 0/3599 [00:00<?, ?it/s]

  0%|          | 2/3599 [00:10<4:14:53,  4.25s/it]

  0%|          | 3/3599 [00:14<4:06:09,  4.11s/it]

  0%|          | 4/3599 [00:19<4:44:13,  4.74s/it]

  0%|          | 5/3599 [00:23<4:21:13,  4.36s/it]

...

100%|█████████▉| 3594/3599 [51:55:54<02:12, 26.54s/it]

100%|█████████▉| 3595/3599 [51:55:59<01:19, 19.99s/it]

100%|█████████▉| 3597/3599 [51:56:05<00:22, 11.16s/it]

100%|█████████▉| 3598/3599 [51:56:11<00:09,  9.47s/it]

100%|██████████| 3599/3599 [51:56:21<00:00, 51.95s/it]


# Staeling even mORE MOOREE DATA FROM YOUTUUBEEEE

In [1]:
import json
from pathlib import Path
import time

crawled_ids = []
with open("logs/additional_videos_crawled.json", "r") as f:
    crawled_ids = json.load(f)
crawled_ids[:3]

[{'id': 'XuXLSagyZcU'}, {'id': 'd3dmIPox05o'}, {'id': 'JjNwVWsIm8U'}]

In [19]:
import os
from yt_dlp import YoutubeDL


def download_video(video_idx: str, output_folder: str = "downloads") -> dict:

    os.makedirs(output_folder, exist_ok=True)
    url = f"https://www.youtube.com/watch?v={video_idx}"

    base_opts = {
        "quiet": True,
        "js_runtimes": {"node": {}}
    }

    try:
        # ---- metadata check ----
        with YoutubeDL(base_opts) as ydl:
            info = ydl.extract_info(url, download=False)

        # Estimated filesize in bytes
        video_filesize = info.get("filesize") or info.get("filesize_approx")
        duration = info.get("duration")

        if duration is None or duration > 1800:
            return {"video_id": video_idx, "error": "video too long"}

        if video_filesize and video_filesize > 500 * 1024 * 1024:
            return {"video_id": video_idx, "error": "video too large"}

        # ---- video download ----
        video_opts = {
            **base_opts,
            "format": "bestvideo[height<=1080][ext=mp4]/bestvideo[height<=1080]/bestvideo/best",
            "outtmpl": os.path.join(output_folder, "%(id)s_video.%(ext)s"),
            "max_filesize": 500 * 1024 * 1024,
            "noplaylist": True,
            "retries": 3,
            "fragment_retries": 3
        }

        with YoutubeDL(video_opts) as ydl:
            ydl.download([url])

        # ---- audio download ----
        audio_opts = {
            **base_opts,
            "format": "bestaudio[ext=m4a]/bestaudio/best",
            "outtmpl": os.path.join(output_folder, "%(id)s_audio.%(ext)s"),
            "postprocessors": [{
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192"
            }],
            "noplaylist": True,
            "retries": 3,
            "fragment_retries": 3
        }

        with YoutubeDL(audio_opts) as ydl:
            ydl.download([url])

        return {
            "video_id": info["id"],
            "title": info.get("title"),
            "video_file": f"{info['id']}_video.mp4",
            "audio_file": f"{info['id']}_audio.mp3"
        }

    except Exception as e:
        return {"video_id": video_idx, "error": str(e)}

In [20]:
output_jsonl = Path("logs/downloaded_crawled_videos.jsonl")
output_jsonl.parent.mkdir(parents=True, exist_ok=True)

total = len(crawled_ids)

with open(output_jsonl, "a", buffering=1) as f:
    for i, item in enumerate(crawled_ids, start=1):

        vid = item["id"]

        result = download_video(
            video_idx=vid,
            output_folder="crawled_downloads"
        )

        if "error" in result:
            record = {
                "id": vid,
                "status": "error",
                "error": result["error"]
            }
        else:
            record = {
                "id": vid,
                "status": "success"
            }

        f.write(json.dumps(record) + "\n")

        if i % 100 == 0:
            print(f"processed {i} / {total}")

        time.sleep(1)

ERROR: [youtube] 7xFlk6YKJYK: Video unavailable
ERROR: [youtube] kSuVatp3xpQ: This video is not available


ERROR: [youtube] RAsYSKjHsOk: This video is not available


ERROR: [youtube] yWHbBD0GuE0: This video is not available


ERROR: [youtube] Oaw3S5AmiRo: This video is not available


[download] Got error: (<HTTPSConnection(host='rr2---sn-a5oxu-uoal.googlevideo.com', port=443) at 0x126317a50>, 'Connection to rr2---sn-a5oxu-uoal.googlevideo.com timed out. (connect timeout=20.0)'). Giving up after 3 retries


...

processed 2000 / 8259                                      


ERROR: [youtube] pWxkPFIBPdA: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


ERROR: [youtube] cpvHbqXu5n4: This video is not available


[download]  12.8% of  489.14MiB at    3.16MiB/s ETA 02:14  

KeyboardInterrupt: 